# Import dependencies

In [1]:
import tensorflow as tf
from transformers import TFBertModel, TFBertTokenizer, AutoTokenizer
from tensorflow.keras.layers import Input, Dense, Dropout, Attention
from tensorflow.keras.models import Model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch

2024-05-13 10:10:08.931294: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-05-13 10:10:08.967960: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-13 10:10:09.556563: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/danish/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_t

In [2]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)

# Prepare Data

In [3]:
file_path = './data.pkl'
df = pd.read_pickle(file_path)
df.head()

,Text,Type,Short Summary
0,دُنیا کی تاریخ زمین اور سمندروں پر ہونے والے ت...,History,سمندر کی گہرائیوں میں پائی جانے والی یہ معدنیا...
1,عام طور پر 5-6 ملی میٹر کی پتھری پیشاب کے ذریع...,Health,تاہم اگر مثانے میں پتھری ہو یا سائز میں چھوٹی ...
2,کوڑے کیسے شروع ہوئے؟\nشروع میں دالوں کو پیس کر...,History,کئی جگہوں پر اس کا ذکر بھی آیا ہے کہ پکوڑے یا ...
3,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔یہ ابت...,Nature,پپیتا نرم و ملائم جلد رکھنے والا پھل ہے۔
4,دل کو تقویت دیتا ہے۔قے اور بخار سے نجات دلاتا ...,General,دل کو تقویت دیتا ہے۔


In [4]:
test_data = df[900:]
data = df[0:900]

In [5]:
from transformers import AutoTokenizer, TFBertModel
import tensorflow as tf

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = TFBertModel.from_pretrained("google-bert/bert-base-uncased")

/home/danish/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-05-13 10:10:14.959458: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9100 MB memory:  -> device: 0, name: NVIDIA RTX A2000 12GB, pci bus id: 0000:65:00.0, compute capability: 8.6
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another

In [6]:
# Function to tokenize the data
def tokenize_and_prepare_masks(data, max_length):
    # Tokenizing the data
    tokens = tokenizer(
        data['Text'].tolist(),
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="np"
    )
    
    return tokens['input_ids'], tokens['attention_mask']

# Maximum sequence length
max_length = 512

# Tokenize data and prepare attention masks
input_ids, attention_masks = tokenize_and_prepare_masks(data, max_length)
input_ids.shape, attention_masks.shape

((900, 512), (900, 512))

In [7]:
def generate_labels(text, summary, tokenizer, max_length):
    tokenized_text = tokenizer.encode(text, add_special_tokens=False)
    tokenized_summary = tokenizer.encode(summary, add_special_tokens=False)
    
    # Initialize labels with 0 (not in summary)
    labels = [0] * len(tokenized_text)
    
    # Set labels to 1 for tokens that appear in the summary
    summary_indices = [i for i, token in enumerate(tokenized_text) if token in tokenized_summary]
    for idx in summary_indices:
        labels[idx] = 1
    
    # Ensure labels match the required max_length
    if len(labels) > max_length:
        labels = labels[:max_length]
    else:
        labels += [0] * (max_length - len(labels))
    
    return labels

# Applying label creation across the dataset
data['labels'] = data.apply(lambda x: generate_labels(x['Text'], x['Short Summary'], tokenizer, max_length), axis=1)

# Convert labels list to appropriate format for training
labels = np.array(data['labels'].tolist())


Token indices sequence length is longer than the specified maximum sequence length for this model (615 > 512). Running this sequence through the model will result in indexing errors
/tmp/ipykernel_2638770/3536866606.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['labels'] = data.apply(lambda x: generate_labels(x['Text'], x['Short Summary'], tokenizer, max_length), axis=1)


In [8]:
from sklearn.model_selection import train_test_split

# Split input_ids and labels
x_train_ids, x_val_ids, y_train, y_val = train_test_split(
    input_ids, labels, test_size=0.1, random_state=42
)

# Split attention_masks using the same indices
_, x_val_masks = train_test_split(
    attention_masks, test_size=0.1, random_state=42
)

# Build and train model

In [11]:
from transformers import TFBertModel
from keras.layers import Input, Dense, Dropout, Attention
from keras.models import Model

def create_extractive_summarization_model(max_length):
    # Define input layers
    input_ids = Input(shape=(max_length,), dtype=tf.int32)
    attention_mask = Input(shape=(max_length,), dtype=tf.int32)
    
    # Load pre-trained BERT model
    bert_model = TFBertModel.from_pretrained('bert-base-uncased')
    
    # BERT-based encoder
    outputs = bert_model(input_ids, attention_mask=attention_mask)[0]
    
    # Attention mechanism
    attention_output = Attention()([outputs, outputs])
    
    # Dense layer
    dense_layer = Dense(units=max_length, activation='relu')(attention_output)
    
    # Output layer
    output = Dense(units=1, activation='sigmoid')(dense_layer)
    
    # Define model
    model = Model(inputs=[input_ids, attention_mask], outputs=output)
    
    return model

# Maximum sequence length
max_length = 512

# Create the model
model = create_extractive_summarization_model(max_length)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()


/home/danish/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initia

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 512)]        0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 512)]        0           []                               
                                                                                                  
 tf_bert_model_1 (TFBertModel)  TFBaseModelOutputWi  109482240   ['input_1[0][0]',                
                                thPoolingAndCrossAt               'input_2[0][0]']                
                                tentions(last_hidde                                               
                                n_state=(None, 512,                                           

In [12]:
tf.__version__

'2.12.0'

In [13]:
!nvidia-smi

Sat May 11 00:50:47 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.06              Driver Version: 545.23.06    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A2000 12GB          On  | 00000000:65:00.0  On |                  Off |
| 30%   49C    P2              23W /  70W |   2552MiB / 12282MiB |      7%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [57]:
history = model.fit(
    [x_train_ids, _], y_train,
    validation_data=([x_val_ids, x_val_masks], y_val),
    epochs=3,
    batch_size=5
)

Epoch 1/3


162/162 [==============================] - 89s 409ms/step - loss: 0.6508 - accuracy: 0.6477 - val_loss: 0.6550 - val_accuracy: 0.6439
Epoch 2/3
162/162 [==============================] - 66s 409ms/step - loss: 0.6526 - accuracy: 0.6477 - val_loss: 0.6597 - val_accuracy: 0.6439
Epoch 3/3
162/162 [==============================] - 68s 418ms/step - loss: 0.6504 - accuracy: 0.6477 - val_loss: 0.6612 - val_accuracy: 0.6439


In [58]:
# Save the model
model_path = "./BERT MODIFIED/"
model.save(model_path+'model.h5')

# Test and Evaluate

In [9]:
import tensorflow as tf
from transformers import TFBertModel

# Register the custom object
custom_objects = {'TFBertModel': TFBertModel}

# Load the model with custom objects
model = tf.keras.models.load_model('./BERT MODIFIED/model.h5', custom_objects=custom_objects)

In [11]:
def preprocess_and_tokenize(text, tokenizer, max_length=512):
    # Tokenizing the single text input
    tokens = tokenizer(
        text,  # Direct text input
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors="tf"  # Ensure this is set to TensorFlow tensors for the model compatibility
    )
    return tokens['input_ids'], tokens['attention_mask']


In [12]:
def generate_summary(input_ids, attention_mask, model, tokenizer):
    # Get model predictions
    predictions = model.predict([input_ids, attention_mask])[0]  # Assuming the model outputs logits

    # Apply threshold to get binary mask
    thresholded_prediction = (predictions > 0.5).astype(int)

    # Convert to flat numpy array if necessary
    thresholded_prediction = thresholded_prediction.flatten()

    # Extract the tokens that are part of the summary
    summary_tokens = [tok for idx, tok in enumerate(input_ids[0]) if thresholded_prediction[idx] == 1]

    # Decode the selected tokens to text
    summary_text = tokenizer.decode(summary_tokens)
    summary_text = summary_text.replace('[PAD]', '')
    summary_text = summary_text.replace('[UNK]', '')
    summary_text = summary_text.replace('#', '')
    summary_text = summary_text.replace('[CLS]', '')
    summary_text = summary_text.replace('[SEP]', '')
    summary_text = summary_text.replace('  ', '')
    return summary_text


In [13]:
def create_summary_from_text(text, tokenizer, model, max_length=512):
    # Preprocess and tokenize the input text
    input_ids, attention_mask = preprocess_and_tokenize(text, tokenizer, max_length)
    
    # Generate the summary
    summary = generate_summary(input_ids, attention_mask, model, tokenizer)
    
    return summary


In [14]:
i = 500
text = data['Text'][i]
orig_sum = data['Short Summary'][i]


user_input_text = text
# Assuming `model` and `tokenizer` are already loaded and available
summary_text = create_summary_from_text(user_input_text, tokenizer, model)
print("Actuall Text", text)
print("Generated Summary:", summary_text)
print("Orignal Summary", orig_sum)

1/1 [==============================] - 2s 2s/step
Actuall Text ڈبلن (اُردو پوائنٹ اخبارتازہ ترین - این این آئی۔ 17 اپریل2024ء) پاکستان کے ٹاپ اسکواش پلیئرز عاصم خان اور نور زمان دو ٹورنامنٹس کھیلنے کیلئے آئرلینڈ پہنچ گئے تاہم ان کا سامان نہ پہنچ سکا، دونوں کے پاس ریکٹس ہیں نہ جوتے ، دونوں کا پہلا میچ بدھ کی شام ہونا ہے۔نور زمان اور عاصم خان پہلے ویسٹ آئرلینڈ اوپن پھر آئرش اوپن میں حصہ لیں گے، ایونٹ میں شرکت کیلئے دونوں پلیئرز پیر کو آئرلینڈ پہنچ گئے تاہم ان کا سامان اب تک نہیں پہنچا۔
12 ہزار ڈالرز انعامی رقم والا ویسٹ آف آئرلینڈ اوپن منگل کو شروع ہوگیا، ایونٹ میں عاصم خان ٹاپ سیڈ ہیں۔ انہیں اور نور زمان کو پہلے راؤنڈ میں بائی ملا جبکہ دوسرا راؤنڈ بدھ کی شام کو ہونا ہے۔دونوں پلیئرز تاحال سامان نہ ملنے کی وجہ سے پریشانی کا شکار ہیں۔


عاصم خان نے بتایا کہ وہ غیر ملکی ایئرلائن سے براستہ استنبول آئرلینڈ پہنچے، فضائی روٹس میں خلل کی وجہ سے فلائٹ کا دورانیہ بڑھ گیا جس کے سبب کنیکٹکنگ فلائٹ میں وقفہ صرف آدھا گھنٹہ بچا تھا جس کی وجہ سے اس فلائٹ پر سامان نہ آسکا۔
عاصم کے مطابق ایئرلائن

2024-05-13 10:10:29.836936: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


In [17]:
orig = []
pred = []
for i in range(900,999):
    summary = create_summary_from_text(test_data['Text'][i], tokenizer,  model)
    orig.append(test_data['Short Summary'][i])
    pred.append(summary)

1/1 [==============================] - 0s 52ms/step


In [34]:
import scores
rouge_scores = scores.calculate_rouge_scores(pred, orig)
print("ROUGE Scores:", rouge_scores)

ROUGE Scores: {'rouge1': 0.06552336552336552, 'rouge2': 0.018902309811400723, 'rougeL': 0.06552336552336552}
